# AI Agent with MCPs using Gemini and PydanticAI

This tutorial uses **Google Gemini** and **Model Context Protocol (MCP)**, an open standard for exposing LLM tools using an API, to build powerful AI agents. The key components for this tutorial are:

- 🤖 **Google Gemini** for state-of-the-art inference
- 🛠️ **PydanticAI** for agent and tool management
- 🔌 **MCP Servers** for prebuilt tool integration

You'll learn how to set up your environment, connect Gemini to real-world tools using MCP, and build a conversational agent capable of reasoning and taking actions.

By the end of this workshop, you'll have built an AI-powered Airbnb assistant agent that can find a place to stay based on preferences like location, budget, and travel dates.

This tutorial includes the following sections:

- [1. Setting up the environment](#step1)
- [2. Installing dependencies](#step2)
- [3. Create a simple instance of a PydanticAI agent](#step3)
- [4. Write a date/time tool for your agent](#step4)
- [5. Replace the date/time tool with an MCP server](#step5)
- [6. Turn your agent into an Airbnb finder](#step6)
- [7. Challenge to expand the agent](#step7)


<a id="step1"></a>

## Step 1: Setting up the environment

Since we are using **Google Gemini** as our LLM backend, there is no need to launch a local inference server. Gemini is accessed via the Google Generative AI API, so all you need is an API key.

### Get your Gemini API Key

1. Go to [Google AI Studio](https://aistudio.google.com/apikey) and create an API key.
2. Store it securely. In this Colab notebook, we'll use Google Colab's **Secrets** feature to manage it safely.


### Using Colab Secrets (Recommended)

1. Click the 🔑 **Key icon** in the left sidebar of Colab.
2. Add a new secret with the name `GOOGLE_API_KEY` and paste your API key as the value.
3. Toggle the **Notebook access** switch to ON.

Then run the cell below to load it into your environment.

In [ ]:
import os

# --- Option 1: Load from Colab Secrets (recommended) ---
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # --- Option 2: Paste your key directly (less secure) ---
    os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

<a id="step2"></a>

## Step 2: Installing dependencies

Install the PydanticAI dependencies with Google Gemini support:

In [ ]:
!pip install -q "pydantic-ai[google]"
!pip install python-dotenv


<a id="step3"></a>

## Step 3: Create a simple instance of a PydanticAI agent

Start by creating a Gemini model instance for your agent.


In [ ]:
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

provider = GoogleProvider(
    api_key=os.environ["GOOGLE_API_KEY"],
)

agent_model = GoogleModel("gemini-2.0-flash", provider=provider)
print("✅ Gemini model initialized.")

Now create an instance of the `Agent` class from `pydantic_ai`. 


In [ ]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

It's time to test the agent. The `pydantic_ai` framework provides multiple ways to run an `Agent` instance. To learn more, see the [PydanticAI site](https://ai.pydantic.dev/agents/#running-agents).

In this workshop, you are running the agent in `async` mode. Define a helper function that allows you to quickly test your agent.

In [ ]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio

async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


Test the agent by calling this function.

In [ ]:
await run_async("What is the capital of France?")

Now that you have the basics of an agent instance, let's connect it to real-world tools.

<a id="step4"></a>

## Step 4: Write a date/time tool for your agent

LLMs rely on their training data to respond to your prompts, so the agent you just defined would fail to answer a factual question that falls outside of its training knowledge. You can show this with an example:

In [ ]:
await run_async("What's the date today?")

It's no surprise that the model failed to answer this question. Now it's time to power-up your LLM by providing the agent with a function that can get the current date. The process by which an LLM triggers a function call is commonly referred to as "Tool Calling" or "Function Calling". In this workshop, you're going to take advantage of the `pydantic_ai` agent `Tool` package to provide the agent with appropriate tools. First, define a custom tool within this framework using the code sample below.

In [ ]:
from datetime import datetime
from pydantic_ai import Tool          

@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


Next, provide this tool to your agent by adding the function signature of the tool to the `Agent` constructor. This notifies the LLM that the new tool exists. 

In [ ]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

Now test the agent.

In [ ]:
await run_async("What's the date today?")

Congratulations on building an agent with access to real-time data. 



<a id="step5"></a>

## Step 5: Replace the date/time tool with an MCP server

Now that you learned how to create a custom tool and let the agent access it, you can enhance this step using the [Model Context Protocol](https://modelcontextprotocol.io/introduction) (MCP). You can replace the custom tool with a simple MCP server that serves the agent and provides similar information.

Why should you use MCP? MCP servers provide:
- ✅ Standardized API interfaces
- 🔄 Reusability across projects
- 📦 Prebuilt functionality

To replace your custom time tool with an official MCP time server, follow these steps:

### Installing an MCP time server

Start by installing the MCP server:


In [ ]:
!pip install -q mcp-server-time

Now define the `time_server`:

In [ ]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
    timeout=60,
)

Finally, modify your agent by removing the previously defined tool and adding the MCP server instead.

In [ ]:
agent = Agent(
    model=agent_model,
    toolsets=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

You can now see whether the agent can use MCP to provide the correct time.

In [ ]:
await run_async("What's the date today?")


You have now used an official MCP server to power up your agent. In the next section, you'll learn how to turn your ideas into real working projects by using the hundreds of available free or paid MCP servers.


<a id="step6"></a>

## Step 6: Turn your agent into an Airbnb finder

As you discovered in the last section, MCP servers are very easy to use. They provide a standard way of providing LLMs with the tools they need. There are already thousands of MCP servers available for you to use. Consult one of the following MCP trackers as a reference to find out about the available servers:
- [https://github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers)
- [https://mcp.so/](https://mcp.so/)

You will use npx to launch your next server. Use the following commands to install the required dependencies:

In [ ]:
# Install Node.js for npx support in Colab
!apt-get update -qq && apt-get install -qq -y nodejs npm > /dev/null 2>&1
!node -v && npm -v && npx --version



In this part of the workshop, you're going to build an agent to help browse available Airbnbs listings to book. You can build on top of what you've done already and add an open-source Airbnb MCP server to your agent. Start by defining the Airbnb server.

In [ ]:
airbnb_server = MCPServerStdio(
    "npx",
    args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"],
    timeout=60,
)

Now update your agent.

In [ ]:
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    toolsets=[time_server, airbnb_server],
    system_prompt=system_prompt,
)


Finally, see if your agent can browse through the Airbnb listings.

In [ ]:
await run_async("Find a place to stay in Vancouver for next Sunday for 3 nights for 2 adults?")



<a id="step7"></a>

## Step 7: Challenge to expand the agent

For an additional challenge, add weather integration using the MCP weather server of your choice:
1. Launch the MCP weather server
2. Add it to the list of agent tools
3. Ask the agent to suggest the best travel dates based on the weather